# Mercedes-Benz Greener Manufacturing — Modeling & Evaluation

**Objective:** ...

**Notebook Outline:** ...

---
## 1. Setup & Data Loading

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn: linear
from sklearn.linear_model import Ridge, Lasso, ElasticNet

# Sklearn: tree-based
from sklearn.ensemble import (RandomForestRegressor,
                               GradientBoostingRegressor,
                               ExtraTreesRegressor)

# Sklearn: pipeline utilities
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (KFold, cross_val_score,
                                      GridSearchCV, RandomizedSearchCV)
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Others
import xgboost as xgb
import lightgbm as lgb
import shap

# Settings
pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
SEED = 42
np.random.seed(SEED)

print('All imports loaded.')

In [ ]:
# Load preprocessed data 
train = pd.read_csv('train_processed.csv')
test  = pd.read_csv('test_processed.csv')

print(f'Train shape : {train.shape}')
print(f'Test  shape : {test.shape}')
train.head(3)

In [ ]:
# Feature/target split 
# Using the train_processed.csv (already preprocessed data). Therefore, 
# all categoricals are one-hot encoded, so no object columns remain, and  
# all features are numeric

feature_cols = [c for c in train.columns if c not in ['ID', 'y']]

X_train = train[feature_cols].values
y_train = train['y'].values
X_test  = test[feature_cols].values

test_ids = test['ID'].values if 'ID' in test.columns else np.arange(len(test))

# Ensuring no object columns remain after preprocessing
obj_cols = train[feature_cols].select_dtypes(include='object').columns.tolist()
assert len(obj_cols) == 0, f"Object columns still present: {obj_cols}"

# Verification that the test has identical columns
assert list(train[feature_cols].columns) == list(test[feature_cols].columns), \
    "Train/test column mismatch!"

print(f"✓ All {len(feature_cols)} features are numeric")
print(f"✓ Train and test columns match")
print(f'X_train : {X_train.shape}')
print(f'y_train : {y_train.shape}')
print(f'X_test  : {X_test.shape}')

---
## 2. Target Variable — (y) Analysis

`y` represents the time (in seconds) a car spends on the test bench. It is right-skewed with outliers. It is modeled with `log(y)` to stabilize the variance and reduce the influence of outliers on the data outputs.  

In [ ]:
# Target distribution: raw vs log 
# Trasformation comparasion 
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(y_train, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Target y  (raw)', fontsize=13)
axes[0].set_xlabel('Seconds on test bench')
axes[0].axvline(np.median(y_train), color='red', linestyle='--', label=f'Median {np.median(y_train):.1f}')
axes[0].legend()

axes[1].hist(np.log1p(y_train), bins=60, color='coral', edgecolor='white')
axes[1].set_title('Target  log1p(y)  (transformed)', fontsize=13)
axes[1].set_xlabel('log1p(seconds)')
axes[1].axvline(np.median(np.log1p(y_train)), color='red', linestyle='--')

plt.tight_layout()
plt.show()

print(f'Raw skewness: {pd.Series(y_train).skew():.3f}')
print(f'Log1p skewness: {pd.Series(np.log1p(y_train)).skew():.3f}')

In [ ]:
# Toggle: use log-transform
USE_LOG = True   # set False to train on raw y

if USE_LOG:
    y_fit = np.log1p(y_train)
    print('Training on log1p(y)')
else:
    y_fit = y_train.copy()
    print('Training on raw y')

---
## 3. Cross-Validation Setup

...

In [ ]:
# ── CV scaffold ─────────────────────────────────────────────────────
kf = KFold(n_splits=10, shuffle=True, random_state=SEED)

def cv_r2(estimator, X, y, log_target=USE_LOG):
    """Return mean ± std CV R² (on original scale if log_target=True)."""
    scores = []
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        estimator.fit(X_tr, y_tr)
        preds = estimator.predict(X_val)
        if log_target:
            preds   = np.expm1(preds)
            y_val_o = np.expm1(y_val)
        else:
            y_val_o = y_val
        scores.append(r2_score(y_val_o, preds))
    return np.mean(scores), np.std(scores)

def cv_rmse(estimator, X, y, log_target=USE_LOG):
    """Return mean ± std CV RMSE on original scale."""
    scores = []
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        estimator.fit(X_tr, y_tr)
        preds = estimator.predict(X_val)
        if log_target:
            preds   = np.expm1(preds)
            y_val_o = np.expm1(y_val)
        else:
            y_val_o = y_val
        scores.append(np.sqrt(mean_squared_error(y_val_o, preds)))
    return np.mean(scores), np.std(scores)

print('CV helpers defined (10-fold KFold, seed=42)')

---
## 4. Baseline Models

...

In [ ]:
# Linear baselines 
# linear regression, ridge regression, kernel regression (RBF),
# regression tree, kernel estimate (RBF), lasso, elasticnet

from sklearn.linear_model import LinearRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.tree import DecisionTreeRegressor

baseline_models = {
    'LinearRegression'    : Pipeline([('scaler', RobustScaler()),
                                       ('model',  LinearRegression())]),

    'Ridge(α=1)'          : Pipeline([('scaler', RobustScaler()),
                                       ('model',  Ridge(alpha=1.0))]),

    'Lasso(α=0.01)'       : Pipeline([('scaler', RobustScaler()),
                                       ('model',  Lasso(alpha=0.01,
                                                        max_iter=5000))]),

    'ElasticNet(α=0.01)'  : Pipeline([('scaler', RobustScaler()),
                                       ('model',  ElasticNet(alpha=0.01,
                                                             l1_ratio=0.5,
                                                             max_iter=5000))]),

    'KernelRidge(RBF)'    : Pipeline([('scaler', RobustScaler()),
                                       ('model',  KernelRidge(alpha=1.0,
                                                              kernel='rbf',
                                                              gamma=0.01))]),

    'KernelEstimate(RBF)' : Pipeline([('scaler', RobustScaler()),
                                       ('model',  KernelRidge(alpha=0.1,
                                                              kernel='rbf',
                                                              gamma=0.001))]),

    'RegressionTree'      : DecisionTreeRegressor(max_depth=5,
                                                   min_samples_leaf=10,
                                                   random_state=SEED),
}

baseline_results = {}
for name, model in baseline_models.items():
    print(f'Evaluating {name}...', end=' ', flush=True)
    r2_mean, r2_std     = cv_r2(model,   X_train, y_fit)
    rmse_mean, rmse_std = cv_rmse(model, X_train, y_fit)
    baseline_results[name] = {'R²': r2_mean, 'R² std': r2_std,
                               'RMSE': rmse_mean, 'RMSE std': rmse_std}
    print(f'R²={r2_mean:.4f}±{r2_std:.4f}  RMSE={rmse_mean:.3f}±{rmse_std:.3f}')

baseline_df = pd.DataFrame(baseline_results).T
display(baseline_df.style.highlight_max(subset=['R²'], color='lightgreen')
                          .highlight_min(subset=['RMSE'], color='lightgreen')
                          .format({'R²':'{:.4f}','R² std':'{:.4f}',
                                   'RMSE':'{:.3f}','RMSE std':'{:.3f}'}))

---
## 5. Tree-Based Models

Tree models handle sparse binary features and non-linear relationships naturally.

- **ExtraTreesRegressor** - ...
- **RandomForestRegressor** - reduce variance via randomization (each tree is trained on a bootstrap sample of data and a random subset of features at each split)
- **GradientBoostingRegressor** - ...

In [ ]:
# ── Tree-based models (default hyperparams first) ───────────────────
tree_models = {
    'ExtraTrees'       : ExtraTreesRegressor(n_estimators=300, n_jobs=-1,
                                              random_state=SEED),
    'RandomForest'     : RandomForestRegressor(n_estimators=300, n_jobs=-1,
                                                random_state=SEED),
    'GradientBoosting' : GradientBoostingRegressor(n_estimators=300,
                                                    learning_rate=0.05,
                                                    max_depth=4,
                                                    subsample=0.8,
                                                    random_state=SEED),
}

tree_results = {}
for name, model in tree_models.items():
    print(f'Evaluating {name}...', end=' ', flush=True)
    r2_mean, r2_std     = cv_r2(model,   X_train, y_fit)
    rmse_mean, rmse_std = cv_rmse(model, X_train, y_fit)
    tree_results[name]  = {'R²': r2_mean, 'R² std': r2_std,
                            'RMSE': rmse_mean, 'RMSE std': rmse_std}
    print(f'R²={r2_mean:.4f}±{r2_std:.4f}  RMSE={rmse_mean:.3f}±{rmse_std:.3f}')

tree_df = pd.DataFrame(tree_results).T
display(tree_df.style.highlight_max(subset=['R²'], color='lightgreen')
                      .highlight_min(subset=['RMSE'], color='lightgreen')
                      .format({'R²':'{:.4f}','R² std':'{:.4f}',
                               'RMSE':'{:.3f}','RMSE std':'{:.3f}'}))

---
## 6. XGBoost / LightGBM 

...

In [ ]:
# XGBoost / LightGBM 
import xgboost as xgb
import lightgbm as lgb

xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.6,
    tree_method='hist', random_state=SEED, n_jobs=-1)

lgb_model = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.6,
    random_state=SEED, n_jobs=-1)

# for name, model in [('XGBoost', xgb_model), ('LightGBM', lgb_model)]:
#     r2_mean, r2_std = cv_r2(model, X_train, y_fit)
#     print(f'{name}: R²={r2_mean:.4f} ± {r2_std:.4f}')

print('XGBoost/LightGBM cells ready — uncomment after pip install.')

---
## 7. ...

---
## 8 ...


---
## 9. ...


---
## 10. ...

---
## 11. ...


---
## 12. ...

---
## 13. Summary & Next Steps